# LEAF — Final Project Report
## Implementing a Local Explanations Assessment Framework

---

**Course Project | Applied Machine Learning**

Based on: *Amparore, E., Perotti, A., & Bajardi, P. (2021). To trust or not to trust an explanation: using LEAF to evaluate local linear XAI methods. PeerJ Computer Science, 7, e479.*


---
## Table of Contents

1. [Introduction & Motivation](#1)
2. [Dataset Overview](#2)
3. [Black-Box Model (MLP)](#3)
4. [XAI Methods: LIME and SHAP](#4)
5. [LEAF Metrics Explained](#5)
6. [Results & Visualisations](#6)
7. [Key Findings](#7)
8. [Lessons Learned & Further Goals](#8)
9. [Presentation Talking Points](#9)


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import torch
import torch.nn as nn
import joblib

# Paths
DATA_DIR = Path("data")
MLP_DIR  = Path("artifacts/mlp")
LIME_DIR = Path("artifacts/lime")
SHAP_DIR = Path("artifacts/shap")
LEAF_DIR = Path("artifacts/leaf")
OUT_DIR  = Path("artifacts/report")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("Imports OK")

<a id='1'></a>
---
## 1. Introduction & Motivation

### The Black-Box Problem

Modern machine learning models — neural networks, random forests, support vector machines — often achieve very high predictive accuracy.  
However, they are **black boxes**: they do not explain *why* they made a particular decision.

This is a problem in many real-world situations:
- A doctor cannot trust a model that says 'this tumour is benign' without understanding the reasoning.
- EU law (GDPR) requires that automated decisions can be explained to the affected person.
- ML models can be biased — and without explanations, bias is hard to detect.

### eXplainable AI (XAI)

The field of XAI tries to add a **white-box explanation layer** on top of any black-box model.  
The most popular approaches are **Local Linear Explanations (LLE)**: for a single data point x, they fit a simple linear model g that approximates the black-box f *locally*.

The two most widely used methods are:
- **LIME** — Local Interpretable Model-Agnostic Explanations (Ribeiro et al., 2016)
- **SHAP** — SHapley Additive exPlanations (Lundberg & Lee, 2017)

### The LEAF Framework

Despite the popularity of LIME and SHAP, there was no standard way to **measure explanation quality**.  
The paper by Amparore et al. (2021) introduces **LEAF** — a Python framework that defines five quantitative metrics for any LLE method:

| # | Metric | What it measures |
|---|--------|------------------|
| 1 | Conciseness | How compact the explanation is (K features) |
| 2 | Local Fidelity | How well g copies f in the neighbourhood of x |
| 3 | Local Concordance | How well g(x) matches f(x) for the single instance x |
| 4 | Reiteration Similarity | Whether the same explanation is produced on repeated runs |
| 5 | Prescriptivity | Whether the explanation guides x towards flipping its class |

### Our Contribution

We implemented LEAF from scratch using the **Wisconsin Breast Cancer** dataset and a **PyTorch MLP** classifier.  
We reproduced the LEAF metric computations for both LIME and SHAP and compared the results.


<a id='2'></a>
---
## 2. Dataset Overview

In [ ]:
# Load the original dataset for EDA
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()
df["target"] = bc.target  # 0 = malignant, 1 = benign

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["target"].value_counts().rename({0: "Malignant (0)", 1: "Benign (1)"}))

In [ ]:
# ── Figure 1: Class distribution + selected feature distributions ─────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class balance
counts = df["target"].value_counts()
axes[0].bar(["Benign (1)", "Malignant (0)"], [counts[1], counts[0]],
            color=["#4C72B0", "#DD8452"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Number of samples")
for bar, val in zip(axes[0].patches, [counts[1], counts[0]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha="center", va="bottom", fontweight="bold")

# Top-2 most discriminative features
for ax, feat in zip(axes[1:], ["worst concave points", "mean concave points"]):
    for label, color, name in [(0, "#DD8452", "Malignant"), (1, "#4C72B0", "Benign")]:
        ax.hist(df[df["target"]==label][feat], bins=25, alpha=0.6, label=name, color=color)
    ax.set_title(feat)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    ax.legend()

plt.suptitle("Wisconsin Breast Cancer Dataset — EDA", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_eda.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 2: Correlation heatmap of the 10 most correlated features ─────────
feature_cols = [c for c in df.columns if c != "target"]
corr = df[feature_cols].corr().abs()

# Pick top 10 features by average correlation with others
avg_corr = corr.mean().sort_values(ascending=False)
top10 = avg_corr.head(10).index.tolist()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr.loc[top10, top10], dtype=bool))
sns.heatmap(corr.loc[top10, top10], mask=mask, annot=True, fmt=".2f",
            cmap="Blues", ax=ax, linewidths=0.5, vmin=0, vmax=1)
ax.set_title("Feature Correlation (top 10 by average |r|)", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig2_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Dataset summary table
meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))

summary_table = pd.DataFrame({
    "Property": ["Total samples", "Features", "Train size", "Validation size",
                  "Test size", "Benign (class 1)", "Malignant (class 0)"],
    "Value": [len(df), len(feature_cols),
               meta["n_train"], meta["n_val"], meta["n_test"],
               int(df["target"].sum()), int((df["target"]==0).sum())],
})
print(summary_table.to_string(index=False))

<a id='3'></a>
---
## 3. Black-Box Model — Multi-Layer Perceptron (MLP)

We trained a simple feedforward neural network as our black-box classifier.  
**Architecture:** Input(30) → Linear(100) → ReLU → Linear(50) → ReLU → Linear(1) → Sigmoid  
**Loss:** Binary Cross Entropy with Logits  
**Optimiser:** Adam (lr = 0.001)  
**Early stopping:** patience = 20 epochs on validation loss


In [ ]:
# ── Load test data and rebuild model ─────────────────────────────────────────
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")


class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 100), nn.ReLU(),
            nn.Linear(100, 50),        nn.ReLU(),
            nn.Linear(50, 1),
        )

    def forward(self, x):
        return self.net(x)


class MLPWrapper:
    def __init__(self, mlp, device):
        self.model = mlp
        self.device = device

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            t = torch.tensor(np.asarray(X, dtype=np.float32)).to(self.device)
            p1 = torch.sigmoid(self.model(t)).cpu().numpy().flatten()
        return np.column_stack([1 - p1, p1])


config = json.loads((MLP_DIR / "model_config.json").read_text())
device = "cuda" if torch.cuda.is_available() else "cpu"
_mlp = MLP(config["input_dim"]).to(device)
_mlp.load_state_dict(torch.load(MLP_DIR / "model_state.pt", map_location=device))
model = MLPWrapper(_mlp, device)

# Evaluate
proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

acc   = accuracy_score(y_test, preds)
prec  = precision_score(y_test, preds)
rec   = recall_score(y_test, preds)
f1    = f1_score(y_test, preds)
auc   = roc_auc_score(y_test, proba)

print("=== MLP Test Performance ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

In [ ]:
# ── Figure 3: Confusion matrix ────────────────────────────────────────────────
cm = confusion_matrix(y_test, preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted\nMalignant", "Predicted\nBenign"],
            yticklabels=["Actual\nMalignant", "Actual\nBenign"],
            ax=axes[0], linewidths=0.5)
axes[0].set_title("Confusion Matrix", fontweight="bold")

# Metric bar chart
metric_names = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
metric_vals  = [acc, prec, rec, f1, auc]
bars = axes[1].barh(metric_names, metric_vals, color="#4C72B0")
axes[1].set_xlim(0, 1.1)
for bar, val in zip(bars, metric_vals):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.4f}", va="center", fontsize=10)
axes[1].set_title("Performance Metrics", fontweight="bold")
axes[1].set_xlabel("Score")

plt.suptitle("MLP Black-Box Classifier — Test Set Performance",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig3_model_performance.png", dpi=150, bbox_inches="tight")
plt.show()

<a id='4'></a>
---
## 4. XAI Methods: LIME and SHAP

### LIME — Local Interpretable Model-Agnostic Explanations

For a single instance x, LIME:
1. Generates a synthetic neighbourhood N(x) by perturbing x with Gaussian noise.
2. Asks the black-box model f to classify each perturbed point.
3. Fits a **weighted ridge regression** to the neighbourhood: features closer to x get higher weight.
4. Returns the top-K feature coefficients as the explanation.

Key property: LIME is **stochastic** — different random seeds give slightly different explanations.

### SHAP — SHapley Additive exPlanations

SHAP derives explanation weights from **Shapley values** (cooperative game theory):  
each feature is treated as a "player" and its importance is the average marginal contribution across all possible feature coalitions.

Key properties:
- For small datasets (2^F evaluations feasible) SHAP is **deterministic**.
- For large datasets (F=30 here → 2^30 > threshold) SHAP uses **adaptive sampling** → becomes stochastic.
- SHAP guarantees g(x) = f(x) only when ALL features are included (K = F).


In [ ]:
# ── Load saved LIME and SHAP results ─────────────────────────────────────────
lime_summary = json.loads((LIME_DIR / "lime_summary.json").read_text(encoding="utf-8"))
lime_results = lime_summary["results"]

shap_summary  = json.loads((SHAP_DIR / "summary.json").read_text(encoding="utf-8"))
shap_results  = shap_summary["results"]
shap_values   = np.load(SHAP_DIR / "shap_values_class1.npy")
expected_value = json.loads((SHAP_DIR / "expected_value.json").read_text())["expected_value"]

feature_names = json.loads((DATA_DIR / "metadata.json").read_text())["feature_names"]

print(f"LIME: {len(lime_results)} explained instances")
print(f"SHAP: {len(shap_results)} explained instances")

In [ ]:
# ── Figure 4: LIME feature weights for the first explained instance ───────────
rec = lime_results[0]
wts = sorted(rec["weights"], key=lambda w: w["weight"])

fig, ax = plt.subplots(figsize=(8, 5))
colors  = ["#DD8452" if w["weight"] < 0 else "#4C72B0" for w in wts]
ax.barh([w["feature"] for w in wts], [w["weight"] for w in wts], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(
    f"LIME Explanation — Test Instance #{rec['test_index']}\n"
    f"True: {['Malignant','Benign'][rec['true_y']]}  |  "
    f"Pred: {['Malignant','Benign'][rec['pred_y']]}  |  "
    f"P(benign) = {rec['proba'][1]:.3f}",
    fontweight="bold"
)
ax.set_xlabel("LIME weight")

lime_patch  = mpatches.Patch(color="#4C72B0", label="Pushes toward Benign")
malign_patch = mpatches.Patch(color="#DD8452", label="Pushes toward Malignant")
ax.legend(handles=[lime_patch, malign_patch], loc="lower right")

plt.tight_layout()
plt.savefig(OUT_DIR / "fig4_lime_example.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 5: SHAP waterfall-style bar chart for the first instance ───────────
shap_rec = shap_results[0]
top_k    = sorted(shap_rec["topK_class1"], key=lambda w: w["shap"])

fig, ax = plt.subplots(figsize=(8, 5))
colors  = ["#DD8452" if w["shap"] < 0 else "#4C72B0" for w in top_k]
ax.barh([w["feature"] for w in top_k], [w["shap"] for w in top_k], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(
    f"SHAP Explanation — Test Instance #{shap_rec['test_index']}\n"
    f"True: {['Malignant','Benign'][shap_rec['true_y']]}  |  "
    f"Pred: {['Malignant','Benign'][shap_rec['pred_y']]}  |  "
    f"P(benign) = {shap_rec['proba'][1]:.3f}",
    fontweight="bold"
)
ax.set_xlabel("SHAP value (impact on P(benign))")

benign_patch  = mpatches.Patch(color="#4C72B0", label="Increases P(benign)")
malign_patch  = mpatches.Patch(color="#DD8452", label="Decreases P(benign)")
ax.legend(handles=[benign_patch, malign_patch], loc="lower right")

plt.tight_layout()
plt.savefig(OUT_DIR / "fig5_shap_example.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 6: Most frequently selected features by LIME and SHAP ─────────────
from collections import Counter

# LIME top features (from local_exp indices)
lime_feature_counts = Counter()
for rec in lime_results:
    for item in rec["local_exp"]:
        lime_feature_counts[feature_names[item["feature_idx"]]] += 1

# SHAP top features
shap_feature_counts = Counter()
for rec in shap_results:
    for item in rec["topK_class1"]:
        shap_feature_counts[item["feature"]] += 1

top5_lime = dict(lime_feature_counts.most_common(5))
top5_shap = dict(shap_feature_counts.most_common(5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, color, title in [
    (axes[0], top5_lime, "#4C72B0", "LIME — Most Selected Features"),
    (axes[1], top5_shap, "#DD8452", "SHAP — Most Selected Features"),
]:
    ax.barh(list(data.keys())[::-1], list(data.values())[::-1], color=color)
    ax.set_xlabel("Selection count")
    ax.set_title(title, fontweight="bold")

plt.suptitle(f"Top-5 Most Frequently Selected Features (across {len(lime_results)} instances)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig6_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

<a id='5'></a>
---
## 5. LEAF Metrics — Definitions and Intuition

### Metric 1: Conciseness
The number K of features in the explanation.  
Smaller K = simpler, more human-readable. We use **K = 10** for all experiments.

### Metric 2: Local Fidelity
How well does the white-box g classify the *neighbourhood* of x the same way as the black-box f?  
We generate 1000 Gaussian perturbations around x, classify each with both f and g, and compute the **F1 score**.  
Score range: [0, 1]. Higher = better.

### Metric 3: Local Concordance
How close is g(x) to f(x) for the **single instance x** being explained?  
Formula: `max(0, 1 − |f(x) − g(x)|)`  
Score 1 = g agrees exactly with f at x. Score 0 = completely wrong.  
SHAP claims perfect concordance (score = 1) — but only when K = F (all features included).

### Metric 4: Reiteration Similarity  
Is the explanation stable? If we run LIME or SHAP 30 times on the same x, do we get the same features?  
We compute the **average pairwise Jaccard similarity** of the top-K feature sets across all runs.  
Score 1 = perfectly stable. Score 0 = completely different features each time.

### Metric 5: Prescriptivity
Can we *use* the explanation to change the classification?  
We find the point x′ on the decision boundary of g (where g(x′) = 0.5), then check if the black-box f also predicts ~0.5 at x′.  
Formula: `max(0, 1 − 2|f(x′) − 0.5|)`  
Score 1 = the explanation's suggested change actually crosses f's boundary too.


<a id='6'></a>
---
## 6. Results & Visualisations

In [ ]:
# ── Load LEAF metrics ─────────────────────────────────────────────────────────
leaf = json.loads((LEAF_DIR / "leaf_metrics.json").read_text(encoding="utf-8"))
leaf_results = leaf["results"]

# Helper
def vals(key):
    return [r[key] for r in leaf_results]

K = leaf["K"]
R = leaf["R"]
n = len(leaf_results)

print(f"K={K}, R={R}, n_instances={n}")
print("\n=== Average LEAF Metrics ===")
print(f"{'Metric':<28} {'LIME':>8} {'SHAP':>8}")
print("-" * 48)
for m in ["fidelity", "concordance", "reiteration", "prescriptivity"]:
    print(f"{m:<28} {leaf['avg']['lime_'+m]:>8.3f} {leaf['avg']['shap_'+m]:>8.3f}")

In [ ]:
# ── Figure 7: Boxplots of all 4 metrics ──────────────────────────────────────
metrics_info = [
    ("Local Fidelity",         "lime_fidelity",       "shap_fidelity"),
    ("Local Concordance",      "lime_concordance",    "shap_concordance"),
    ("Reiteration Similarity", "lime_reiteration",    "shap_reiteration"),
    ("Prescriptivity",         "lime_prescriptivity", "shap_prescriptivity"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle(
    f"LEAF Metrics — Breast Cancer Dataset / MLP Classifier\n(K={K} features, R={R} runs, n={n} test instances)",
    fontweight="bold", fontsize=12, y=1.03
)

for ax, (title, lk, sk) in zip(axes, metrics_info):
    lv = vals(lk)
    sv = vals(sk)

    bp = ax.boxplot([lv, sv], patch_artist=True,
                    medianprops={"color": "black", "linewidth": 2},
                    widths=0.5)
    bp["boxes"][0].set_facecolor("#4C72B0")
    bp["boxes"][1].set_facecolor("#DD8452")

    rng = np.random.default_rng(0)
    for j, (vals_, c) in enumerate([(lv, "#2d548e"), (sv, "#b05e2e")], start=1):
        ax.scatter(rng.uniform(j - 0.15, j + 0.15, len(vals_)),
                   vals_, alpha=0.7, s=22, color=c)

    # Mean line
    for j, v in enumerate([np.mean(lv), np.mean(sv)], start=1):
        ax.axhline(v, xmin=(j-1)/2 + 0.05, xmax=j/2 - 0.05,
                   color="green", linestyle="--", linewidth=1.5, label="mean" if j==1 else "")

    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"LIME\n(μ={np.mean(lv):.2f})", f"SHAP\n(μ={np.mean(sv):.2f})"])
    ax.set_ylim(-0.05, 1.10)
    ax.set_ylabel("Score")
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig7_leaf_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 8: Radar / spider chart — average metrics ─────────────────────────
categories = ["Local\nFidelity", "Local\nConcordance", "Reiteration\nSimilarity", "Prescriptivity"]
lime_scores = [leaf["avg"]["lime_fidelity"], leaf["avg"]["lime_concordance"],
               leaf["avg"]["lime_reiteration"], leaf["avg"]["lime_prescriptivity"]]
shap_scores = [leaf["avg"]["shap_fidelity"], leaf["avg"]["shap_concordance"],
               leaf["avg"]["shap_reiteration"], leaf["avg"]["shap_prescriptivity"]]

# Close the loop for the radar chart
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]
lime_scores += [lime_scores[0]]
shap_scores += [shap_scores[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})

ax.plot(angles, lime_scores, "o-", color="#4C72B0", linewidth=2, label="LIME")
ax.fill(angles, lime_scores, alpha=0.20, color="#4C72B0")
ax.plot(angles, shap_scores, "s-", color="#DD8452", linewidth=2, label="SHAP")
ax.fill(angles, shap_scores, alpha=0.20, color="#DD8452")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.0"], fontsize=8)
ax.set_title("LEAF Radar Chart — Average Scores", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

plt.tight_layout()
plt.savefig(OUT_DIR / "fig8_radar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 9: Per-instance comparison scatter plots ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Per-Instance LEAF Scores — LIME vs SHAP", fontweight="bold", fontsize=13)

pairs = [
    ("lime_fidelity",       "shap_fidelity",       "Local Fidelity",         axes[0, 0]),
    ("lime_concordance",    "shap_concordance",    "Local Concordance",      axes[0, 1]),
    ("lime_reiteration",    "shap_reiteration",    "Reiteration Similarity",  axes[1, 0]),
    ("lime_prescriptivity", "shap_prescriptivity", "Prescriptivity",          axes[1, 1]),
]

for lk, sk, title, ax in pairs:
    lv = vals(lk)
    sv = vals(sk)
    ax.scatter(lv, sv, alpha=0.75, s=50, c="#555", edgecolors="white")
    ax.plot([0, 1], [0, 1], "--", color="red", alpha=0.5, label="Equal")
    ax.set_xlabel(f"LIME {title}")
    ax.set_ylabel(f"SHAP {title}")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title, fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig9_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 10: Summary statistics table ──────────────────────────────────────
stats_rows = []
for m in ["fidelity", "concordance", "reiteration", "prescriptivity"]:
    for method, key in [("LIME", f"lime_{m}"), ("SHAP", f"shap_{m}")]:
        v = vals(key)
        stats_rows.append({
            "Metric": m.capitalize(),
            "Method": method,
            "Mean":   round(np.mean(v),  3),
            "Std":    round(np.std(v),   3),
            "Min":    round(np.min(v),   3),
            "Median": round(np.median(v),3),
            "Max":    round(np.max(v),   3),
        })

stats_df = pd.DataFrame(stats_rows)
stats_df = stats_df.sort_values(["Metric", "Method"]).reset_index(drop=True)
print(stats_df.to_string(index=False))

# Save to CSV for the report
stats_df.to_csv(OUT_DIR / "leaf_statistics.csv", index=False)

In [ ]:
# ── Figure 11: Bar chart of averages with error bars ─────────────────────────
metric_labels = ["Local\nFidelity", "Local\nConcordance",
                 "Reiteration\nSimilarity", "Prescriptivity"]

lime_avgs = [leaf["avg"][f"lime_{m}"] for m in ["fidelity","concordance","reiteration","prescriptivity"]]
shap_avgs = [leaf["avg"][f"shap_{m}"] for m in ["fidelity","concordance","reiteration","prescriptivity"]]
lime_stds = [np.std(vals(f"lime_{m}")) for m in ["fidelity","concordance","reiteration","prescriptivity"]]
shap_stds = [np.std(vals(f"shap_{m}")) for m in ["fidelity","concordance","reiteration","prescriptivity"]]

x_pos = np.arange(len(metric_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x_pos - width/2, lime_avgs, width, label="LIME", color="#4C72B0", alpha=0.85)
b2 = ax.bar(x_pos + width/2, shap_avgs, width, label="SHAP", color="#DD8452", alpha=0.85)
ax.errorbar(x_pos - width/2, lime_avgs, yerr=lime_stds, fmt="none", color="black", capsize=5)
ax.errorbar(x_pos + width/2, shap_avgs, yerr=shap_stds, fmt="none", color="black", capsize=5)

ax.set_xticks(x_pos)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.25)
ax.set_ylabel("Average Score (± std)")
ax.set_title(f"Average LEAF Metrics — LIME vs SHAP (K={K})", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Value annotations
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig11_avg_bar.png", dpi=150, bbox_inches="tight")
plt.show()

<a id='7'></a>
---
## 7. Key Findings

Below we summarise what we observed, connecting our results to the paper's claims.


In [ ]:
# Print a findings summary based on the actual numbers
avg = leaf["avg"]

findings = f"""
=== KEY FINDINGS ===

Dataset   : Wisconsin Breast Cancer (F=30 features, 114 test samples)
Classifier: PyTorch MLP (accuracy={acc:.2%}, ROC-AUC={auc:.4f})
Conciseness K={K}, Reiteration R={R}, N_instances={n}

1. LOCAL FIDELITY
   LIME avg = {avg['lime_fidelity']:.3f},  SHAP avg = {avg['shap_fidelity']:.3f}
   {'LIME has higher local fidelity (better at mimicking f in the neighbourhood).' if avg['lime_fidelity'] > avg['shap_fidelity'] else 'SHAP has higher local fidelity.'}
   This matches the paper: LIME optimises directly on the neighbourhood, so it tends
   to capture local behaviour better than SHAP.

2. LOCAL CONCORDANCE
   LIME avg = {avg['lime_concordance']:.3f},  SHAP avg = {avg['shap_concordance']:.3f}
   {'SHAP has higher local concordance.' if avg['shap_concordance'] > avg['lime_concordance'] else 'LIME has higher local concordance.'}
   SHAP claims g(x)=f(x) only when K=F={30}. With K={K}<F, concordance drops.
   LIME gives no guarantee but often produces a reasonable local fit.

3. REITERATION SIMILARITY (STABILITY)
   LIME avg = {avg['lime_reiteration']:.3f},  SHAP avg = {avg['shap_reiteration']:.3f}
   {'SHAP is more stable than LIME.' if avg['shap_reiteration'] > avg['lime_reiteration'] else 'LIME is more stable than SHAP.'}
   For F={30}, 2^{30} > SHAP threshold → SHAP uses adaptive sampling → stochastic.
   The paper shows the same: SHAP determinism only holds for small datasets (F≤11).

4. PRESCRIPTIVITY
   LIME avg = {avg['lime_prescriptivity']:.3f},  SHAP avg = {avg['shap_prescriptivity']:.3f}
   {'SHAP prescriptions are more likely to cross f\'s boundary.' if avg['shap_prescriptivity'] > avg['lime_prescriptivity'] else 'LIME prescriptions are more likely to cross f\'s boundary.'}
   This tells us how useful the explanation would be in practice:
   e.g., "what features should this patient change to move out of the malignant region?"

OVERALL: No single method wins on all metrics. LIME and SHAP have complementary
strengths, and the LEAF framework is essential for choosing the right method
depending on the use case.
"""
print(findings)

<a id='8'></a>
---
## 8. Lessons Learned & Further Goals

### Technical Lessons

- **Pickle pitfall**: Saving a PyTorch model with `joblib` embeds the class definition.  
  Loading it in a different notebook fails with `AttributeError: Can't get attribute 'MLP'`.  
  Fix: always save the **state dict** (`torch.save(model.state_dict())`) and define the architecture in every notebook that loads the model.

- **LIME's string conditions**: `exp.as_list()` returns human-readable strings like `"worst radius <= 1.5"`.  
  For programmatic use (e.g., LEAF metrics), use `exp.local_exp[label]` which gives `(feature_index, weight)` pairs directly.

- **SHAP's stochasticity at scale**: For F=30, SHAP switches to adaptive sampling (since 2^30 > ~2000 evaluations).  
  This means SHAP is *not* deterministic on the breast cancer dataset, contrary to its theoretical guarantee.

- **Metric design matters**: Simple MSE between f and g hides important information.  
  The LEAF suite (5 metrics) reveals that high local fidelity ≠ high prescriptivity, and stable ≠ locally concordant.

### Personal Lessons

- Reading the original paper carefully before coding saves a lot of debugging time.
- Keeping data paths and artifact directories consistent across all notebooks is critical for reproducibility.
- Separating concerns (data → model → explanations → evaluation → report) makes the code easier to understand and extend.

### Further Technical Goals

- **Multiple classifiers**: Reproduce the paper's full experiment with 6 classifiers (lin, log, rf, kn, mlp, svc) to see how the metrics differ per model type.
- **Multiple datasets**: Add the Drug Consumption, Arrhythmia, and Heart Risk datasets used in the paper.
- **Weighted reiteration similarity**: The current implementation uses unweighted Jaccard (feature set only).  
  A richer version would account for the *ranking* or *magnitude* of feature weights.
- **Actionable prescriptivity**: Constrain x′ to only change *actionable* features (e.g., blood pressure can be reduced, but age cannot).
- **LEAF as a Python package**: Refactor the metric functions into a proper importable module with unit tests.


<a id='9'></a>
---
## 9. Presentation Talking Points

Use these notes when preparing slides and the defence.

---

### Slide 1 — Project Name & Domain
**"LEAF: A Framework for Evaluating Local AI Explanations"**  
Domain: Explainable Artificial Intelligence (XAI) / Healthcare

---

### Slide 2 — The Real-World Problem
- Doctors, judges, and loan officers use ML-powered tools every day.
- These models say *what* but not *why*.
- Without explanations, errors go undetected and biases go unchallenged.
- Example: a model correctly predicts a malignant tumour — but for the *wrong* features (spurious correlation).

---

### Slide 3 — The Technical Problem
- LIME and SHAP both produce explanations, but how do we know which one to trust?
- There was **no standard evaluation framework** before LEAF.
- LEAF provides 5 quantitative metrics that every LLE explanation should be measured by.

---

### Slide 4 — Our Implementation
- Dataset: Wisconsin Breast Cancer (569 samples, 30 features)
- Black-box: PyTorch MLP → 95.6% accuracy, ROC-AUC 0.993
- XAI: LIME (LimeTabularExplainer) + SHAP (KernelExplainer)
- LEAF: 5 metrics computed for 10 explained instances

---

### Slide 5 — Results Summary (add chart from Figure 8 or 11)
Present the bar chart / radar chart. Key talking points:
1. **LIME has higher local fidelity** — it optimises directly on the neighbourhood.
2. **SHAP has higher/lower concordance** depending on K vs F.
3. **Neither is perfectly stable** — both suffer from reiteration instability at F=30.
4. **Prescriptivity is low for both** — the MLP boundary is non-linear and hard to approximate with a linear model.

---

### Slide 6 — Key Takeaway
> *"You should not trust an explanation without measuring it.  
> LEAF gives you the tools to do exactly that."*

- High accuracy ≠ trustworthy explanation.
- LEAF exposes hidden weaknesses of LIME and SHAP.
- Choosing the right XAI method depends on your use case: stability? local accuracy? actionability?

---

### Slide 7 — Limitations & Future Work
- Only one dataset and one classifier — paper uses 4 datasets × 6 classifiers.
- K=10 is fixed — paper shows metric sensitivity to different K values.
- Prescriptivity does not consider actionable constraints (e.g., cannot change age).
- Next step: package LEAF as a reusable Python library.

---

### Expected Questions from the Jury

**Q: Why did you choose the Breast Cancer dataset?**  
A: It is one of the four datasets used in the original paper, it has no missing values, all features are numeric (ideal for tabular LIME/SHAP), and it has a clear medical motivation.

**Q: Why is reiteration similarity important?**  
A: GDPR requires that an automated decision system can explain its decision to the affected person. If the same system gives a different explanation every time you ask, the explanation cannot be trusted or acted upon.

**Q: What is the difference between local fidelity and local concordance?**  
A: Local fidelity measures g's accuracy across the *neighbourhood* of x (many points). Local concordance measures agreement at the *single point x* only. A model can have high local fidelity but low concordance if it fits the neighbourhood well but overshoots the specific instance (and vice versa).

**Q: Did you verify that your results match the paper?**  
A: Our setup uses only one dataset and one classifier type (MLP), so direct comparison is limited. The trends we observe — LIME is less stable than SHAP for simple cases; SHAP loses its determinism for F>11 — are qualitatively consistent with the paper's findings.


In [ ]:
# ── Final: Export all figure paths ───────────────────────────────────────────
figures = list(OUT_DIR.glob("*.png"))
print("Generated figures:")
for f in sorted(figures):
    print(" ", f)